In [1]:
import sys
import os
import numpy as np
import scipy.linalg as la
from scipy.signal import cont2discrete
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D 
import matplotlib as mpl

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.config as cfg
from utils.quadcopter_model import quad_hover_linear_model, create_discrete_model, gaussian_dist, visualize_gaussian_samples, animate_trajectories, plot_angles, plot_xyz, plot_controls_grid, plot_angles_mean, plot_xyz_mean, plot_controls_grid_mean, summarize_checks
from utils.simple_MLP import PolicyRNN , PolicyMLP, rollout_policy, traj_cost, train_nn_policy, evaluate_policy_cost, plot_loss_evolution

# Plotting parameters
plt.style.use('default')
plt.rcParams['figure.dpi'] = 400

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"], 
    "axes.unicode_minus": False 
})

# Consistent printing of numpy arrays
np.set_printoptions(precision=3, suppress=True)

In [2]:
# Create and discretize model
A, B, C, D = quad_hover_linear_model()
# print("A =\n", A, "\n\nB =\n", B, "\n\nC =\n", C, "\n\nD =\n", D)

A_d, B_tilde_d, C_d, D_d = create_discrete_model(A, B, C, D)
B_d = B_tilde_d[:, :4]
# print("A_d =\n", A_d, "\n\nB_tilde_d =\n", B_tilde_d, "\n\nC_d =\n", C_d, "\n\nD_d =\n", D_d)

In [3]:
# Sanity check
summarize_checks("Hover (CT)", A, B, C, discrete=False)
summarize_checks("Hover (DT)", A_d, B_d, C_d, discrete=True)

# CT system is marginally stable (not asymptotically stable): linearized around hover, 
# so positions and angles have integrator dynamics (they don’t decay by themselves).
# DT: discretizing integrators, we get poles at z=1, so still marginally stable.


=== Hover (CT) (continuous) ===
n = 12
Eigenvalues: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Asymptotically stable: False
Controllable: True (rank 12/12)
Reachability index: 4
Observable: True (rank 12/12)
Stabilizable (PBH): True
Detectable (PBH): True

=== Hover (DT) (discrete) ===
n = 12
Eigenvalues: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Asymptotically stable: False
Controllable: True (rank 12/12)
Reachability index: 4
Observable: True (rank 12/12)
Stabilizable (PBH): True
Detectable (PBH): True


In [4]:
# Training points

# Generate disturbance wind samples for training
train_dist_mean_x  = 4.0
train_dist_sigma_x = 0.0001
nb_training_points  = 1

dist_x_train = gaussian_dist(train_dist_mean_x, train_dist_sigma_x, nb_training_points, device="cpu")           # x-axis disturbance (random wind per trajectory, constant over time)
dist_y_train = torch.zeros_like(dist_x_train)                                                                   # y-axis disturbance (zero)
dist_z_train = -cfg.G * torch.ones_like(dist_x_train)                                                           # z-axis disturbance (gravity)
disturbances_train = torch.cat([dist_x_train, dist_y_train, dist_z_train], dim=-1)                              # [B, T, 3] # [B, T, 3] (B = number of trajectories (batch points), T = number of timesteps per trajectory, 3 = scalar disturbance value for that axis)

visualize_gaussian_samples(dist_x_train, train_dist_mean_x, train_dist_sigma_x, nb_training_points, "single_point_training_dist_samples.png")

# Initial state conditions for training
x0_train_np = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]*nb_training_points)
x0_train = torch.from_numpy(x0_train_np).float()

In [5]:
# TRAINING
device = "cpu"
epochs = 3500
compute_mean_every = 200
lr = 1e-3

policy, train_means, eval_means = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=x0_train, disturbances=disturbances_train,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=False
)

[ep 200/3500]:   train_loss=2.9973e+04 ± 0.00e+00   |   eval_loss=2.7429e+04 ± 0.00e+00   |   |x_T|_avg=75.163
[ep 400/3500]:   train_loss=2.5811e+04 ± 0.00e+00   |   eval_loss=3.3055e+04 ± 0.00e+00   |   |x_T|_avg=28.989
[ep 600/3500]:   train_loss=1.6791e+03 ± 0.00e+00   |   eval_loss=1.3801e+03 ± 0.00e+00   |   |x_T|_avg=6.709
[ep 800/3500]:   train_loss=3.4465e+02 ± 0.00e+00   |   eval_loss=3.4731e+02 ± 0.00e+00   |   |x_T|_avg=3.131
[ep 1000/3500]:   train_loss=1.5486e+02 ± 0.00e+00   |   eval_loss=1.5557e+02 ± 0.00e+00   |   |x_T|_avg=0.580
[ep 1200/3500]:   train_loss=1.2604e+02 ± 0.00e+00   |   eval_loss=1.3116e+02 ± 0.00e+00   |   |x_T|_avg=1.146
[ep 1400/3500]:   train_loss=1.1086e+02 ± 0.00e+00   |   eval_loss=1.1279e+02 ± 0.00e+00   |   |x_T|_avg=0.207
[ep 1600/3500]:   train_loss=9.7851e+01 ± 0.00e+00   |   eval_loss=9.7546e+01 ± 0.00e+00   |   |x_T|_avg=0.081
[ep 1800/3500]:   train_loss=9.3748e+01 ± 0.00e+00   |   eval_loss=9.2273e+01 ± 0.00e+00   |   |x_T|_avg=0.094
[ep

In [6]:
# Plot loss evolution
plot_loss_evolution(train_means, eval_means, epochs, compute_mean_every, filename="single_point_losses.png")

In [7]:
# TESTING on training points
with torch.no_grad():
    X_training, U_training, mean_cost_training, std_cost_training, costs_training = evaluate_policy_cost(A_d, B_tilde_d, policy, x0_train, disturbances_train, device=device)

print(f"Training cost:  {mean_cost_training:.3e} ± {std_cost_training:.2e}")

plot_xyz_mean(X_training, filename="single_point_training_xyz.png")
plot_angles_mean(X_training, filename="single_point_training_angles.png")
plot_controls_grid_mean(U_training, filename="single_point_training_controls.png")
animate_trajectories(X_training, "single_point_training_xyz_3d.mp4")

Training cost:  8.529e+01 ± 0.00e+00
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [8]:
# Testing points

# Generate disturbance wind samples for testing
test_dist_mean_x  = -5.0
test_dist_sigma_x = 1
nb_testing_points  = 50

dist_x_test = gaussian_dist(test_dist_mean_x, test_dist_sigma_x, nb_testing_points, device="cpu")     # x-axis disturbance (random wind per trajectory, constant over time)
dist_y_test = torch.zeros_like(dist_x_test)                                                           # y-axis disturbance (zero)
dist_z_test = -cfg.G * torch.ones_like(dist_x_test)                                                   # z-axis disturbance (gravity)
disturbances_test = torch.cat([dist_x_test, dist_y_test, dist_z_test], dim=-1)                        # [B, T, 3] # [B, T, 3] (B = number of trajectories (batch points), T = number of timesteps per trajectory, 3 = scalar disturbance value for that axis)

visualize_gaussian_samples(dist_x_test, test_dist_mean_x, test_dist_sigma_x, nb_testing_points, "single_point_testing_dist_samples.png")

# Initial state conditions for testing
x0_test_np = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]*nb_testing_points)
x0_test = torch.from_numpy(x0_test_np).float()

In [9]:
# TESTING on testing points
with torch.no_grad():
    X_testing, U_testing, mean_cost_testing, std_cost_testing, costs_testing = evaluate_policy_cost(A_d, B_tilde_d, policy, x0_test, disturbances_test, device=device)

print(f"Testing cost:  {mean_cost_testing:.3e} ± {std_cost_testing:.2e}")

plot_xyz_mean(X_testing, filename="single_point_testing_xyz_N1.png")
plot_angles_mean(X_testing, filename="single_point_testing_angles_N1.png")
plot_controls_grid_mean(U_testing, filename="single_point_testing_controls_N1.png")
animate_trajectories(X_testing, "single_point_testing_xyz_3d_N1.mp4")

Testing cost:  2.639e+07 ± 9.33e+06
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [10]:
# Cost comparison
pct_increase = (mean_cost_testing - mean_cost_training) / mean_cost_training * 100
print(f"Cost increase from training to testing: {pct_increase:.2f} %")

Cost increase from training to testing: 30937291.97 %
